# Multi-Horizon Forecasting for Limit Order Books: Novel Deep Learning Approaches and Hardware Acceleration using Intelligent Processing Units
#### Authors: Zihao Zhang and Stefan Zohren
#### Oxford-Man Institute of Quantitative Finance, Department of Engineering Science, University of Oxford

This jupyter notebook is used to demonstrate our recent paper [2] published in <...>. We use FI-2010 [1] dataset and present how model architecture is constructed here. The FI-2010 is publicly avilable and interested readers can check out their paper [1]. 

#### Data:
The FI-2010 is publicly avilable and interested readers can check out their paper [1]. The dataset can be downloaded from: https://etsin.fairdata.fi/dataset/73eb48d7-4dbc-4a10-a52a-da745b47a649 

Otherwise, the notebook will download the data automatically or it can be obtained from: 

https://drive.google.com/drive/folders/1Xen3aRid9ZZhFqJRgEMyETNazk02cNmv?usp=sharing.

#### References:

[1] Ntakaris A, Magris M, Kanniainen J, Gabbouj M, Iosifidis A. Benchmark dataset for mid‐price forecasting of limit order book data with machine learning methods. Journal of Forecasting. 2018 Dec;37(8):852-66. https://arxiv.org/abs/1705.03233

[2] Zhang Z, Zohren S. Multi-Horizon Forecasting for Limit Order Books: Novel Deep Learning Approaches and Hardware Acceleration using Intelligent Processing Units. 


## Libs 

In [1]:
import os
import torch

import logging
import glob
import argparse
import sys
import time
import pandas as pd
import pickle
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import accuracy_score, classification_report
import torch.nn.functional as F


In [33]:
# obtain data
# Dowload data if they are not existed, nothing otherwise 
if not os.path.isfile('data.zip'):
    !wget https://raw.githubusercontent.com/zcakhaa/DeepLOB-Deep-Convolutional-Neural-Networks-for-Limit-Order-Books/master/data/data.zip
    !unzip -n data.zip
    print('data downloaded.')
else:
    print('data already existed.')

data already existed.


In [3]:
# Check if GPUs are available
if torch.cuda.is_available():
    try:
        # Set memory growth for each GPU
        for i in range(torch.cuda.device_count()):
            torch.cuda.set_per_process_memory_fraction(0.8, i)
        print(torch.cuda.device_count(), "Physical GPUs,", torch.cuda.device_count(), "Logical GPUs")
    except RuntimeError as e:
        print(e)
else:
    print("No GPUs found.")


No GPUs found.


In [4]:
# set random seeds
SEED = 1

np.random.seed(SEED)
torch.manual_seed(SEED)


## Preprocess 

In [18]:
# Import keras
import keras
import tensorflow as tf

In [38]:
# Given imported data of shape (149, 25000) it returns (25000, 40)
# 149 are possible features 
# 25000 is the number of samples
# 40 is the number of features to be used
def prepare_x(data): 
    df1 = data[:40, :].T 
    return np.array(df1)

# Example of usage 
data = np.random.rand(149, 2000) # generate random data of size (40, 5)
x = prepare_x(data) # prepare the input data

print(f"data shape: {data.shape}") # print the shape of the input data
print(f"x shape: {x.shape}") # print the shape of the input data


data shape: (149, 2000)
x shape: (2000, 40)
data shape: (149, 5000)
x shape: (5000, 40)


In [21]:
# This function is used to prepare the output data for the model
# The data has size (40, 5) where 40 is the number of features and 5 is the number of time steps
# Return the output data of size (5, 5, 3), where 5 is the number of time steps, 5 is the number of features, and 3 is the number of classes
def get_label(data): 
    lob = data[-5:, :].T 
    all_label = []

    for i in range(lob.shape[1]): # this for loop is used to prepare the output data for the model
        one_label = lob[:, i] - 1
        one_label = keras.utils.to_categorical(one_label, 3)
        one_label = one_label.reshape(len(one_label), 1, 3)
        all_label.append(one_label)

    return np.hstack(all_label) 

# Example of usage 
data = np.random.rand(40,5) # generate random data of size (40, 5)
y = get_label(data) # prepare the output data

print(f"data shape: {data.shape}") # print the shape of the input data
print(f"y shape: {y.shape}") # print the shape of the output data
print(f"\ny \n {y}") # print the output data


data shape: (40, 5)
y shape: (5, 5, 3)

y 
 [[[1. 0. 0.]
  [1. 0. 0.]
  [1. 0. 0.]
  [1. 0. 0.]
  [1. 0. 0.]]

 [[1. 0. 0.]
  [1. 0. 0.]
  [1. 0. 0.]
  [1. 0. 0.]
  [1. 0. 0.]]

 [[1. 0. 0.]
  [1. 0. 0.]
  [1. 0. 0.]
  [1. 0. 0.]
  [1. 0. 0.]]

 [[1. 0. 0.]
  [1. 0. 0.]
  [1. 0. 0.]
  [1. 0. 0.]
  [1. 0. 0.]]

 [[1. 0. 0.]
  [1. 0. 0.]
  [1. 0. 0.]
  [1. 0. 0.]
  [1. 0. 0.]]]


In [14]:
# This function is used to prepare the input data for the model
def data_classification(X, Y, T): 
    [N, D] = X.shape
    df = np.array(X)

    dY = np.array(Y)

    dataY = dY[T - 1:N]

    dataX = np.zeros((N - T + 1, T, D))
    for i in range(T, N + 1):
        dataX[i - T] = df[i - T:i, :]

    return dataX.reshape(dataX.shape + (1,)), dataY 

# Example of usage
X = np.random.rand(40, 5) # generate random input data of size (40, 5)
Y = np.random.rand(5, 5, 3) # generate random output data of size (5, 5, 3)
T = 5 # set the number of time steps    

X, Y = data_classification(X, Y, T) # prepare the input and output data

# X has 36 samples, 5 time steps, 5 features (chosen in the 40 features given), and 1 channel
# Y has 1 sample, 5 time steps, and 3 classes
print(f"X shape: {X.shape}") # print the shape of the input data
print(f"Y shape: {Y.shape}") # print the shape of the output data
print(f"\nX \n {X}") # print the input data
print(f"\nY \n {Y}") # print the output data


X shape: (36, 5, 5, 1)
Y shape: (1, 5, 3)

X 
 [[[[0.07531087]
   [0.42572441]
   [0.60200784]
   [0.42540347]
   [0.81072123]]

  [[0.94188625]
   [0.86991077]
   [0.7039995 ]
   [0.47029892]
   [0.68269804]]

  [[0.01341831]
   [0.98247812]
   [0.17207733]
   [0.9062604 ]
   [0.8071099 ]]

  [[0.80013968]
   [0.26893606]
   [0.21915381]
   [0.42880069]
   [0.8493711 ]]

  [[0.45037964]
   [0.93542011]
   [0.35267933]
   [0.61939217]
   [0.98069904]]]


 [[[0.94188625]
   [0.86991077]
   [0.7039995 ]
   [0.47029892]
   [0.68269804]]

  [[0.01341831]
   [0.98247812]
   [0.17207733]
   [0.9062604 ]
   [0.8071099 ]]

  [[0.80013968]
   [0.26893606]
   [0.21915381]
   [0.42880069]
   [0.8493711 ]]

  [[0.45037964]
   [0.93542011]
   [0.35267933]
   [0.61939217]
   [0.98069904]]

  [[0.64346252]
   [0.04549822]
   [0.46355709]
   [0.33475134]
   [0.53218155]]]


 [[[0.01341831]
   [0.98247812]
   [0.17207733]
   [0.9062604 ]
   [0.8071099 ]]

  [[0.80013968]
   [0.26893606]
   [0.21915381]

In [27]:
# This function creates the dataset for the model
# x_train is the input data of size (36, 5, 40) where 36 is the number of samples, 5 is the number of time steps, and 40 is the number of features
# y_train is the output data of size (36, 5, 1, 3), where 36 is the number of samples, 5 is the number of time steps, 1 is the number of features, and 3 is the number of classes
# batch_size is the batch size
# method is the method to create the dataset
# shuffle is a boolean variable to shuffle the dataset
def create_dataset(x_train, y_train, batch_size, method, shuffle=False):
    train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
    if shuffle:
        train_ds = train_ds.shuffle(len(x_train))
    train_ds = train_ds.batch(batch_size, drop_remainder=True)
    train_ds = train_ds.map(lambda d, l: (tf.cast(d, tf.float32), tf.cast(l, tf.float32)))

    if method == 'train':
        return train_ds.repeat()

    if method == 'val':
        return train_ds

    if method == 'test':
        return train_ds

    if method == 'prediction':
        train_ds = tf.data.Dataset.from_tensor_slices((x_train))
        train_ds = train_ds.batch(batch_size, drop_remainder=True)
        train_ds = train_ds.map(lambda d: (tf.cast(d, tf.float32)))
        return train_ds
    

# Example of usage
x_train = np.random.rand(36, 5, 40) # generate random input data of size (36, 5, 40)
y_train = np.random.rand(36, 5, 1, 3) # generate random output data of size (36, 5, 1, 3)
batch_size = 4 # set the batch size
method = 'train' # set the method to create the dataset
shuffle = False # shuffle the dataset

train_ds = create_dataset(x_train, y_train, batch_size, method, shuffle) # create the dataset
print(f"train_ds: {train_ds}") # print the dataset



train_ds: <_RepeatDataset element_spec=(TensorSpec(shape=(4, 5, 40), dtype=tf.float32, name=None), TensorSpec(shape=(4, 5, 1, 3), dtype=tf.float32, name=None))>


In [28]:
# This function is used to evaluate the model
# real_y is the real output data of size (36, 5, 1, 3), where 36 is the number of samples, 5 is the number of time steps, 1 is the number of features, and 3 is the number of classes
# pred_y is the predicted output data of size (36, 5, 1, 3), where 36 is the number of samples, 5 is the number of time steps, 1 is the number of features, and 3 is the number of classes
def evaluation_metrics(real_y, pred_y):
    real_y = real_y[:len(pred_y)]
    logging.info('-------------------------------')

    for i in range(real_y.shape[1]):
        print(f'Prediction horizon = {i}')
        print(f'accuracy_score = {accuracy_score(np.argmax(real_y[:, i], axis=1), np.argmax(pred_y[:, i], axis=1))}')
        print(f'classification_report = {classification_report(np.argmax(real_y[:, i], axis=1), np.argmax(pred_y[:, i], axis=1), digits=4)}')
        print('-------------------------------')

# Example of usage
real_y = np.random.rand(36, 5, 1, 3) # generate random real output data of size (36, 5, 1, 3)
pred_y = np.random.rand(36, 5, 1, 3) # generate random predicted output data of size (36, 5, 1, 3)

evaluation_metrics(real_y, pred_y) # evaluate the model





Prediction horizon = 0
accuracy_score = 1.0
classification_report =               precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000         0
           1     0.0000    0.0000    0.0000         0
           2     0.0000    0.0000    0.0000         0

   micro avg     0.0000    0.0000    0.0000         0
   macro avg     0.0000    0.0000    0.0000         0
weighted avg     0.0000    0.0000    0.0000         0
 samples avg     0.0000    0.0000    0.0000         0

-------------------------------
Prediction horizon = 1
accuracy_score = 1.0
classification_report =               precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000         0
           1     0.0000    0.0000    0.0000         0
           2     0.0000    0.0000    0.0000         0

   micro avg     0.0000    0.0000    0.0000         0
   macro avg     0.0000    0.0000    0.0000         0
weighted avg     0.0000    0.0000    0.0000         0
 samples avg     

/home/samuele/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/samuele/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/samuele/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/samuele/.local

In [30]:
# This function is used to prepare the input data for the decoder
# data is the output data of size (36, 5, 1, 3), where 36 is the number of samples, 5 is the number of time steps, 1 is the number of features, and 3 is the number of classes
# teacher_forcing is a boolean variable to use teacher forcing
def prepare_decoder_input(data, teacher_forcing):
    if teacher_forcing:
        first_decoder_input = keras.utils.to_categorical(np.zeros(len(data)), 3)
        first_decoder_input = first_decoder_input.reshape(len(first_decoder_input), 1, 3)
        decoder_input_data = np.hstack((data[:, :-1, :], first_decoder_input))

    if not teacher_forcing:
        decoder_input_data = np.zeros((len(data), 1, 3))
        decoder_input_data[:, 0, 0] = 1.

    # The output is the input data for the decoder of size (36, 5, 3), where 36 is the number of samples, 5 is the number of time steps, and 3 is the number of classes
    return decoder_input_data

# Example of usage
data = np.random.rand(36, 5, 1, 3) # generate random output data of size (36, 5, 1, 3)
teacher_forcing = False # use teacher forcing

decoder_input_data = prepare_decoder_input(data, teacher_forcing) # prepare the input data for the decoder

print(f"data shape: {data.shape}") # print the shape of the output data
print(f"decoder_input_data shape: {decoder_input_data.shape}") # print the shape of the input data for the decoder
print(f"\ndecoder_input_data \n {decoder_input_data}") # print the input data for the decoder



data shape: (36, 5, 1, 3)
decoder_input_data shape: (36, 1, 3)

decoder_input_data 
 [[[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]

 [[1. 0. 0.]]]


## Manipulate data

In [31]:
# please change the data_path to your local path
# data_path = '/nfs/home/zihaoz/limit_order_book/data'
T = 50
epochs = 50
batch_size = 32
n_hidden = 64
checkpoint_filepath = './model_deeplob_seq/weights'

In [36]:
# %%
dec_train = np.loadtxt('Train_Dst_NoAuction_DecPre_CF_7.txt')
dec_test1 = np.loadtxt('Test_Dst_NoAuction_DecPre_CF_7.txt')
dec_test2 = np.loadtxt('Test_Dst_NoAuction_DecPre_CF_8.txt')
dec_test3 = np.loadtxt('Test_Dst_NoAuction_DecPre_CF_9.txt')
dec_test = np.hstack((dec_test1, dec_test2, dec_test3))

# Print shapes
print(f"Dec_train shape:{dec_train.shape}")
print(f"Dec_test1 shape:{dec_test1.shape}")
print(f"Dec_test2 shape:{dec_test2.shape}")
print(f"Dec_test3 shape:{dec_test3.shape}")
print(f"Dec_test shape:{dec_test.shape}")




Dec_train shape:(149, 254750)
Dec_test1 shape:(149, 55478)
Dec_test2 shape:(149, 52172)
Dec_test3 shape:(149, 31937)
Dec_test shape:(149, 139587)


In [42]:
# extract limit order book data from the FI-2010 dataset
# The first element will be the samples 
# The second element will be the number of features used
train_lob = prepare_x(dec_train)
test_lob = prepare_x(dec_test)

# Print shapes
print(f"Dec_train shape:{dec_train.shape}") 
print(f"Dec_test shape:{dec_test.shape}")

print(f"\nTrain_lob shape:{train_lob.shape}") 
print(f"Test_lob shape:{test_lob.shape}")


Dec_train shape:(149, 254750)
Dec_test shape:(149, 139587)

Train_lob shape:(254750, 40)
Test_lob shape:(139587, 40)


In [43]:
# extract label from the FI-2010 dataset
# The first element will be the samples
# The second element will be the number of horizons
# The third element will be the number of classes
train_label = get_label(dec_train)
test_label = get_label(dec_test)

# Print shapes
print(f"Dec_train shape:{dec_train.shape}")
print(f"Dec_test shape:{dec_test.shape}")

print(f"Train_label shape:{train_label.shape}")
print(f"Test_label shape:{test_label.shape}")




Dec_train shape:(149, 254750)
Dec_test shape:(149, 139587)
Train_label shape:(254750, 5, 3)
Test_label shape:(139587, 5, 3)


In [44]:
# prepare training data. We feed past 100 observations into our algorithms.
# Input:
# train_lob: the training data
# train_label: the training label
# T: window size (T = 50 means are taken 50 LOB to predicts then next LOB (in our case 5 since the data is 5))
# Output:
# The first element will be the input data for the encoder
# The second element will be the output data for the decoder
train_encoder_input, train_decoder_target = data_classification(train_lob, train_label, T)

# Print shapes
print(f"Train_lob shape:{train_lob.shape}")
print(f"Train_label shape:{train_label.shape}")
print(f"T: {T}")

print(f"\nTrain_encoder_input shape:{train_encoder_input.shape}")
print(f"Train_decoder_target shape:{train_decoder_target.shape}")


Train_lob shape:(254750, 40)
Train_label shape:(254750, 5, 3)
T: 50
Train_encoder_input shape:(254701, 50, 40, 1)
Train_decoder_target shape:(254701, 5, 3)


In [45]:
train_decoder_input = prepare_decoder_input(train_encoder_input, teacher_forcing=False)

# Print shapes
print(f"Train_encoder_input shape:{train_encoder_input.shape}")
print(f"Train_decoder_input shape:{train_decoder_input.shape}")






Train_encoder_input shape:(254701, 50, 40, 1)
Train_decoder_input shape:(254701, 1, 3)


In [46]:
test_encoder_input, test_decoder_target = data_classification(test_lob, test_label, T)

# Print shapes
print(f"Test_lob shape:{test_lob.shape}")
print(f"Test_label shape:{test_label.shape}")
print(f"T: {T}")

print(f"\nTest_encoder_input shape:{test_encoder_input.shape}")
print(f"Test_decoder_target shape:{test_decoder_target.shape}")



Test_lob shape:(139587, 40)
Test_label shape:(139587, 5, 3)
T: 50

Test_encoder_input shape:(139538, 50, 40, 1)
Test_decoder_target shape:(139538, 5, 3)


In [47]:
test_decoder_input = prepare_decoder_input(test_encoder_input, teacher_forcing=False)

# Print shapes
print(f"Test_encoder_input shape:{test_encoder_input.shape}")

print(f"Test_decoder_input shape:{test_decoder_input.shape}")

Test_encoder_input shape:(139538, 50, 40, 1)
Test_decoder_input shape:(139538, 1, 3)


## Model

Model in keras

In [48]:
# Input shape = (36, 50, 40, 1)
def get_model_seq(latent_dim):

    # Cho 在 2014 年提出了 Encoder–Decoder 结构，即由两个 RNN 组成，
    # https://arxiv.org/pdf/1406.1078.pdf

    # 50 is the window 
    # 40 is the number of features
    # 1 is the number of channels
    input_train = keras.layers.Input(shape=(50, 40, 1))

    # First convolution
    # 32 is the number of filters
    # (1, 2) is the kernel size
    # (1, 2) is the strides
    conv_first1 = keras.layers.Conv2D(32, (1, 2), strides=(1, 2))(input_train)
    conv_first1 = keras.layers.LeakyReLU(negative_slope=0.01)(conv_first1)
    
    conv_first1 = keras.layers.Conv2D(32, (4, 1), padding='same')(conv_first1)
    conv_first1 = keras.layers.LeakyReLU(negative_slope=0.01)(conv_first1)
    conv_first1 = keras.layers.Conv2D(32, (4, 1), padding='same')(conv_first1)
    conv_first1 = keras.layers.LeakyReLU(negative_slope=0.01)(conv_first1)

    conv_first1 = keras.layers.Conv2D(32, (1, 2), strides=(1, 2))(conv_first1)
    conv_first1 = keras.layers.LeakyReLU(negative_slope=0.01)(conv_first1)
    conv_first1 = keras.layers.Conv2D(32, (4, 1), padding='same')(conv_first1)
    conv_first1 = keras.layers.LeakyReLU(negative_slope=0.01)(conv_first1)
    conv_first1 = keras.layers.Conv2D(32, (4, 1), padding='same')(conv_first1)
    conv_first1 = keras.layers.LeakyReLU(negative_slope=0.01)(conv_first1)

    conv_first1 = keras.layers.Conv2D(32, (1, 10))(conv_first1)
    conv_first1 = keras.layers.LeakyReLU(negative_slope=0.01)(conv_first1)
    conv_first1 = keras.layers.Conv2D(32, (4, 1), padding='same')(conv_first1)
    conv_first1 = keras.layers.LeakyReLU(negative_slope=0.01)(conv_first1)
    conv_first1 = keras.layers.Conv2D(32, (4, 1), padding='same')(conv_first1)
    conv_first1 = keras.layers.LeakyReLU(negative_slope=0.01)(conv_first1)

    # build the inception module
    convsecond_1 = keras.layers.Conv2D(64, (1, 1), padding='same')(conv_first1)
    convsecond_1 = keras.layers.LeakyReLU(negative_slope=0.01)(convsecond_1)
    convsecond_1 = keras.layers.Conv2D(64, (3, 1), padding='same')(convsecond_1)
    convsecond_1 = keras.layers.LeakyReLU(negative_slope=0.01)(convsecond_1)

    convsecond_2 = keras.layers.Conv2D(64, (1, 1), padding='same')(conv_first1)
    convsecond_2 = keras.layers.LeakyReLU(negative_slope=0.01)(convsecond_2)
    convsecond_2 = keras.layers.Conv2D(64, (5, 1), padding='same')(convsecond_2)
    convsecond_2 = keras.layers.LeakyReLU(negative_slope=0.01)(convsecond_2)

    convsecond_3 = keras.layers.MaxPooling2D((3, 1), strides=(1, 1), padding='same')(conv_first1)
    convsecond_3 = keras.layers.Conv2D(64, (1, 1), padding='same')(convsecond_3)
    convsecond_3 = keras.layers.LeakyReLU(negative_slope=0.01)(convsecond_3)

    convsecond_output = keras.layers.concatenate([convsecond_1, convsecond_2, convsecond_3], axis=3)

    conv_reshape = keras.layers.Reshape((int(convsecond_output.shape[1]), int(convsecond_output.shape[3])))(convsecond_output)

    encoder_inputs = conv_reshape
    print("Encoder inputs shape:", encoder_inputs.shape)

    encoder = keras.layers.LSTM(latent_dim, return_state=True)
    encoder_outputs, state_h, state_c = encoder(encoder_inputs)
    print("Encoder LSTM outputs shape:", encoder_outputs.shape)

    decoder_inputs = keras.layers.Input(shape=(1, 3))
    print("Decoder inputs shape:", decoder_inputs.shape)

    all_outputs = []
    encoder_outputs = keras.layers.Reshape((1, int(encoder_outputs.shape[1])))(encoder_outputs)
    inputs = keras.layers.concatenate([decoder_inputs, encoder_outputs], axis=2)
    print("Initial inputs shape for decoder LSTM:", inputs.shape)

    decoder_lstm = keras.layers.LSTM(latent_dim, return_sequences=True, return_state=True)
    decoder_dense = keras.layers.Dense(3, activation='softmax')

    for _ in range(5):
        outputs, state_h, state_c = decoder_lstm(inputs, initial_state=[state_h, state_c])
        print("Decoder LSTM outputs shape:", outputs.shape)

        outputs = decoder_dense(keras.layers.concatenate([outputs, encoder_outputs], axis=2))
        print("Decoder dense outputs shape:", outputs.shape)

        all_outputs.append(outputs)
        inputs = keras.layers.concatenate([outputs, encoder_outputs], axis=2)
        print("Updated inputs shape for next step:", inputs.shape)

    decoder_outputs = keras.layers.Lambda(lambda x: K.concatenate(x, axis=1))(all_outputs)
    print("Final decoder outputs shape:", decoder_outputs.shape)

    model = keras.models.Model([input_train, decoder_inputs], decoder_outputs)
    return model

Model of Joel

In [9]:
import torch
import torch.nn as nn

class LobCNN(nn.Module):
    def __init__(   self,
                    lookback_period = 100,
                    inner_convolutional_layers = 2,
                    smoothing_percentage = 0.06,
                    convolution_channels = 32,
                    inception_channels = 64,
                    lstm_layers = 2,
                    output_size = 1,
                    regression = True,
                ):
        super(LobCNN, self).__init__()

        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.name = 'LobCNN'

        self.regression = regression
        self.lstm_layers = lstm_layers
        self.output_size = output_size
        self.convolution_channels = convolution_channels
        self.inception_channels = inception_channels
        self.smoothing_kernel_size = int(smoothing_percentage * lookback_period/inner_convolutional_layers) + 1

        # convolution blocks
        modules_list = [
            nn.Conv2d(  in_channels=1, 
                        out_channels=self.convolution_channels, 
                        kernel_size=(1, 2), 
                        stride=(1, 2)),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(self.convolution_channels),
        ]
        for _ in range(inner_convolutional_layers):
            modules_list.append(nn.Conv2d(  in_channels=self.convolution_channels, 
                                            out_channels=self.convolution_channels, 
                                            kernel_size=(self.smoothing_kernel_size, 1),
                                            stride=(1, 1)))
            modules_list.append(nn.LeakyReLU(negative_slope=0.01))
            modules_list.append(nn.BatchNorm2d(self.convolution_channels))
        self.conv1 = nn.Sequential(*modules_list)
        
        modules_list = [
            nn.Conv2d(  in_channels=self.convolution_channels, 
                        out_channels=self.convolution_channels, 
                        kernel_size=(1, 2), 
                        stride=(1, 2)),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(self.convolution_channels),
        ]
        for _ in range(inner_convolutional_layers):
            modules_list.append(nn.Conv2d(  in_channels=self.convolution_channels, 
                                            out_channels=self.convolution_channels, 
                                            kernel_size=(self.smoothing_kernel_size, 1),
                                            stride=(1, 1)))
            modules_list.append(nn.LeakyReLU(negative_slope=0.01))
            modules_list.append(nn.BatchNorm2d(self.convolution_channels))
        self.conv2 = nn.Sequential(*modules_list)

        modules_list = [
            nn.Conv2d(  in_channels=self.convolution_channels, 
                        out_channels=self.convolution_channels, 
                        kernel_size=(1, 10), 
                        stride=(1, 1)),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(self.convolution_channels),
        ]
        for _ in range(inner_convolutional_layers):
            modules_list.append(nn.Conv2d(  in_channels=self.convolution_channels, 
                                            out_channels=self.convolution_channels, 
                                            kernel_size=(self.smoothing_kernel_size, 1), 
                                            stride=(1, 1)))
            modules_list.append(nn.LeakyReLU(negative_slope=0.01))
            modules_list.append(nn.BatchNorm2d(self.convolution_channels))
        self.conv3 = nn.Sequential(*modules_list)

        # Inception modules: smoothing the time series with different scales.
        # A large kernel size is used to capture a global distribution of the image 
        # while a small kernel size is used to capture more local information.
        # Inception network architecture makes it possible to use filters of multiple 
        # sizes without increasing the depth of the network. The different filters 
        # are added parallelly instead of being fully connected one after the other.
        # The following scheme is the same as the one of the original paper of 
        # googleLeNet: https://www.cs.unc.edu/~wliu/papers/GoogLeNet.pdf .
        self.inception_1 = nn.Sequential(
            nn.Conv2d(in_channels=self.convolution_channels, out_channels=self.inception_channels, kernel_size=(1, 1), padding='same'),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(self.inception_channels),
            nn.Conv2d(in_channels=self.inception_channels, out_channels=self.inception_channels, kernel_size=(3, 1), padding='same'),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(self.inception_channels),
        )
        self.inception_2 = nn.Sequential(
            nn.Conv2d(in_channels=self.convolution_channels, out_channels=self.inception_channels, kernel_size=(1, 1), padding='same'),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(self.inception_channels),
            nn.Conv2d(in_channels=self.inception_channels, out_channels=self.inception_channels, kernel_size=(5, 1), padding='same'),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(self.inception_channels),
        )
        self.inception_3 = nn.Sequential(
            nn.MaxPool2d((3, 1), stride=(1, 1), padding=(1, 0)),
            nn.Conv2d(in_channels=self.convolution_channels, out_channels=self.inception_channels, kernel_size=(1, 1), padding='same'),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(self.inception_channels),
        )
        self.inception_4 = nn.Sequential(
            nn.Conv2d(in_channels=self.convolution_channels, out_channels=self.inception_channels, kernel_size=(1, 1), padding='same'),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(self.inception_channels),
        )

        # RNN or lstm layer(s)
        self.dropout = nn.Dropout(0.1)
        self.gru = nn.GRU(input_size=self.inception_channels*4, hidden_size=self.inception_channels, num_layers=elf.lstm_layers, batch_first=True)
        self.lstm = nn.LSTM(input_size=self.inception_channels*4, hidden_size=self.inception_channels, num_layers=self.lstm_layers, batch_first=True)
        self.linear_out = nn.Linear(self.inception_channels, self.output_size)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)

        x_inception_1 = self.inception_1(x)
        x_inception_2 = self.inception_2(x)
        x_inception_3 = self.inception_3(x)
        x_inception_4 = self.inception_4(x)

        x = torch.cat((x_inception_1, x_inception_2, x_inception_3, x_inception_4), dim=1)
        x = x.permute(0, 2, 1, 3)
        x = torch.reshape(x, (-1, x.shape[1], x.shape[2]))
        x = self.dropout(x)

        h0 = torch.zeros(self.lstm_layers, x.size(0), self.inception_channels).to(self.device)
        c0 = torch.zeros(self.lstm_layers, x.size(0), self.inception_channels).to(self.device)
        x, _ = self.lstm(x, (h0, c0))
        x = x[:, -1, :]
        x = self.linear_out(x)
        if self.regression:
            #x = torch.squeeze(x)
            return x
        else:
            forecast_y = torch.softmax(x, dim=1)
            return forecast_y
        
        # fixed params sizes
        #torch.Size([32, 1, 100, 40])
        #torch.Size([32, 32, 94, 20])
        #torch.Size([32, 32, 88, 10])
        #torch.Size([32, 32, 82, 1])
        #torch.Size([32, 64, 82, 1])
        #torch.Size([32, 64, 82, 1])
        #torch.Size([32, 64, 82, 1])
        #torch.Size([32, 192, 82, 1])
        #torch.Size([32, 82, 192])
        #torch.Size([32, 64])
        #torch.Size([32, 1])

        ## variable params sizes
        #torch.Size([32, 1, 100, 40])
        #torch.Size([32, 32, 94, 20])
        #torch.Size([32, 32, 88, 10])
        #torch.Size([32, 32, 82, 1])
        #torch.Size([32, 64, 82, 1])
        #torch.Size([32, 64, 82, 1])
        #torch.Size([32, 64, 82, 1])
        #torch.Size([32, 64, 82, 1])
        #torch.Size([32, 256, 82, 1])
        #torch.Size([32, 82, 256])
        #torch.Size([32, 64])
        #torch.Size([32, 1])


In [11]:
# Define a lobCN instance
model = LobCNN()

NameError: name 'elf' is not defined

In [49]:
# Define the model 
model = get_model_seq(n_hidden)
model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer='adam')

print(f"Model defined with n_hidden: {n_hidden}")

Encoder inputs shape: (None, 50, 192)
Encoder LSTM outputs shape: (None, 64)
Decoder inputs shape: (None, 1, 3)


ValueError: A `Concatenate` layer requires inputs with matching shapes except for the concatenation axis. Received: input_shape=[(None, 1, 3), (None, 1, 64)]

In [ ]:
split_train_val = int(np.floor(len(train_encoder_input) * 0.8))

model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_loss',
    mode='auto',
    save_best_only=True)

model.fit([train_encoder_input[:split_train_val], train_decoder_input[:split_train_val]], 
          train_decoder_target[:split_train_val],
          validation_data=([train_encoder_input[split_train_val:], train_decoder_input[split_train_val:]], 
          train_decoder_target[split_train_val:]),
          epochs=epochs, batch_size=batch_size, verbose=2, callbacks=[model_checkpoint_callback])

Epoch 1/50
6368/6368 - 181s - loss: 0.9576 - accuracy: 0.5171 - val_loss: 0.8496 - val_accuracy: 0.6138
Epoch 2/50
6368/6368 - 179s - loss: 0.6975 - accuracy: 0.7035 - val_loss: 0.7852 - val_accuracy: 0.6524
Epoch 3/50
6368/6368 - 180s - loss: 0.6446 - accuracy: 0.7311 - val_loss: 0.7602 - val_accuracy: 0.6728
Epoch 4/50
6368/6368 - 179s - loss: 0.6162 - accuracy: 0.7455 - val_loss: 0.7625 - val_accuracy: 0.6723
Epoch 5/50
6368/6368 - 180s - loss: 0.5984 - accuracy: 0.7540 - val_loss: 0.7443 - val_accuracy: 0.6825
Epoch 6/50
6368/6368 - 181s - loss: 0.5848 - accuracy: 0.7603 - val_loss: 0.7305 - val_accuracy: 0.6892
Epoch 7/50
6368/6368 - 180s - loss: 0.5740 - accuracy: 0.7649 - val_loss: 0.7427 - val_accuracy: 0.6822
Epoch 8/50
6368/6368 - 180s - loss: 0.5650 - accuracy: 0.7689 - val_loss: 0.7615 - val_accuracy: 0.6819
Epoch 9/50
6368/6368 - 179s - loss: 0.5576 - accuracy: 0.7719 - val_loss: 0.7527 - val_accuracy: 0.6846
Epoch 10/50
6368/6368 - 179s - loss: 0.5518 - accuracy: 0.7745 -

In [ ]:
model.load_weights(checkpoint_filepath)
pred = model.predict([test_encoder_input, test_decoder_input])

In [ ]:
evaluation_metrics(test_decoder_target, pred)


Prediction horizon = 0
accuracy_score = 0.8222921354756411
classification_report =               precision    recall  f1-score   support

           0     0.6910    0.5771    0.6289     21147
           1     0.8593    0.9309    0.8937     98624
           2     0.7136    0.5427    0.6165     19767

    accuracy                         0.8223    139538
   macro avg     0.7546    0.6835    0.7130    139538
weighted avg     0.8131    0.8223    0.8143    139538

-------------------------------
Prediction horizon = 1
accuracy_score = 0.7423497541888231
classification_report =               precision    recall  f1-score   support

           0     0.6292    0.5130    0.5652     27448
           1     0.7868    0.8924    0.8363     86605
           2     0.6454    0.4793    0.5501     25485

    accuracy                         0.7423    139538
   macro avg     0.6871    0.6283    0.6505    139538
weighted avg     0.7300    0.7423    0.7307    139538

-------------------------------
Predicti